# Imports

In [1]:
import pandas as pd
import altair as alt
import janitor
import requests
import json
from pathlib import Path
import numpy as np

alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")

RendererRegistry.enable('default')

# Data

In [2]:
FUEL_COLS = {
    "electricity_use_kbtu_": "Electricity",
    "natural_gas_use_kbtu_": "Natural gas",
    "district_steam_use_kbtu_": "District steam",
    "district_chilled_water_use_kbtu_": "District chilled water",
    "all_other_fuel_use_kbtu_": "Other fuel",
}

def chi_bench_df():
    chi_bench_df = pd.read_csv('data/Chicago_Energy_Benchmarking_20261001.csv', thousands=",")
    # normalize column names
    clean_df = chi_bench_df.clean_names()
    # normalize community area names
    clean_df['community_area'] = clean_df['community_area'].str.title()
    # Two types of reporting status are used for submitted building
    clean_df = clean_df[clean_df["reporting_status"].isin(["Submitted", "Submitted Data"])] # reduces from 28329 to 5621 rows
    for c in [
        "chicago_energy_rating", "latitude", "longitude", "year_built", "gross_floor_area_buildings_sq_ft_", "energy_star_score",
        "site_eui_kbtu_sq_ft_", "source_eui_kbtu_sq_ft_",
        "total_ghg_emissions_metric_tons_co2e_", "ghg_intensity_kg_co2e_sq_ft_", *FUEL_COLS.keys(),
    ]:
        clean_df[c] = pd.to_numeric(clean_df[c], errors="coerce")
    return clean_df

print(chi_bench_df().columns)
chi_bench_df()

Index(['data_year', 'id', 'property_name', 'reporting_status', 'address',
       'zip_code', 'chicago_energy_rating',
       'exempt_from_chicago_energy_rating', 'community_area',
       'primary_property_type', 'gross_floor_area_buildings_sq_ft_',
       'year_built', '#_of_buildings', 'water_use_kgal_', 'energy_star_score',
       'electricity_use_kbtu_', 'natural_gas_use_kbtu_',
       'district_steam_use_kbtu_', 'district_chilled_water_use_kbtu_',
       'all_other_fuel_use_kbtu_', 'site_eui_kbtu_sq_ft_',
       'source_eui_kbtu_sq_ft_', 'weather_normalized_site_eui_kbtu_sq_ft_',
       'weather_normalized_source_eui_kbtu_sq_ft_',
       'total_ghg_emissions_metric_tons_co2e_', 'ghg_intensity_kg_co2e_sq_ft_',
       'latitude', 'longitude', 'location', 'row_id'],
      dtype='str')


,data_year,id,property_name,reporting_status,address,zip_code,chicago_energy_rating,exempt_from_chicago_energy_rating,community_area,primary_property_type,...,site_eui_kbtu_sq_ft_,source_eui_kbtu_sq_ft_,weather_normalized_site_eui_kbtu_sq_ft_,weather_normalized_source_eui_kbtu_sq_ft_,total_ghg_emissions_metric_tons_co2e_,ghg_intensity_kg_co2e_sq_ft_,latitude,longitude,location,row_id
3,2022,256888,NaN,Submitted Data,318 N CARPENTER ST,60607,4.0,False,NaN,Office,...,32.4,61.9,32.8,61.4,311.5,3.0,41.887365,-87.653505,"(41.88736471232649, -87.653505317143)",2022-256888
18,2022,256882,NaN,Submitted Data,1648 W DIVISION ST,60622,1.5,False,NaN,Multifamily Housing,...,105.0,138.1,108.6,139.8,638.8,7.0,41.903424,-87.669405,"(41.903424405382026, -87.66940457779671)",2022-256882
24,2022,160184,29 E. Madison LLC,Submitted Data,29 E Madison St,60602,2.5,False,Loop,Office,...,117.0,203.5,122.7,209.4,2361.5,9.8,41.881965,-87.626793,"(41.881965330252534, -87.62679322159418)",2022-160184
25,2022,134776,University Club of Chicago,Submitted Data,76 E Monroe St,60603,1.0,False,Loop,Hotel,...,277.1,453.0,282.4,455.2,3277.6,22.8,41.880924,-87.624934,"(41.88092403509479, -87.62493419985192)",2022-134776
28,2020,252064,Mansueto Library,Submitted Data,1100 E 57th St,60637,2.0,False,Hyde Park,Library,...,240.8,323.6,246.0,329.9,1160.9,18.1,41.792213,-87.599950,"(41.79221307, -87.59994981)",2020-252064
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28323,2023,252311,Frances Larry Apartments,Submitted,824-32 E 53rd St,60615,4.0,NaN,Hyde Park,Multifamily Housing,...,52.1,68.7,57.9,75.0,211.9,3.5,41.910843,-87.629022,"(41.910843, -87.629022)",2023-252311
28325,2023,101753,222 South Riverside,Submitted,222 South Riverside Plaza,60606,4.0,NaN,Near West Side,Office,...,56.1,111.8,60.1,116.0,6936.2,5.6,41.874634,-87.630418,"(41.874634, -87.630418)",2023-101753
28326,2023,175987,CH10 - Chicago South,Submitted,11840 S Marshfield Ave,60643,2.5,NaN,Morgan Park,Office,...,64.1,126.8,65.6,131.0,820.2,6.4,41.973128,-87.651076,"(41.973128, -87.651076)",2023-175987
28327,2023,250083,Daniel Hudson Burnham Apartments,Submitted,1930 W. Loyola Ave,60626,2.5,NaN,West Ridge,Multifamily Housing,...,105.4,151.3,117.5,163.2,757.9,7.6,41.893483,-87.621931,"(41.893483, -87.621931)",2023-250083


# Hardcoded variables

In [3]:
# Chicago community areas geojson
GEOJSON_URL = "https://data.cityofchicago.org/resource/igwz-8jzy.geojson"
def load_chicago_geojson(path="data/chicago_community_areas.geojson"):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text())
    r = requests.get(GEOJSON_URL, timeout=30)
    r.raise_for_status()
    gj = r.json()
    if "features" not in gj:
        raise ValueError(f"Unexpected response from {GEOJSON_URL}: {str(gj)[:200]}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(gj))
    return gj

CHI_GJ = load_chicago_geojson()

In [4]:
# Get dataset with top types from the latest year
# Each property can show up multiple times for each report year. Grab records from the latest year
LATEST_YEAR = chi_bench_df()["data_year"].max()
latest = chi_bench_df()[chi_bench_df()["data_year"] == LATEST_YEAR].copy()

# There are 45 different types of properties
# data centers are top 24 with count 5
type_counts = (latest["primary_property_type"]
               .value_counts()
               .rename_axis("primary_property_type")
               .reset_index(name="properties"))
print(type_counts)

TOP_N_TYPES = 24
top_types = latest["primary_property_type"].value_counts().head(TOP_N_TYPES).index.tolist()
latest_top = latest[latest["primary_property_type"].isin(top_types)]

                         primary_property_type  properties
0                          Multifamily Housing        1355
1                                  K-12 School         351
2                                       Office         332
3                           College/University          84
4                                        Hotel          76
5                      Senior Living Community          52
6                    Supermarket/Grocery Store          42
7                                 Retail Store          37
8                           Mixed Use Property          33
9                                   Strip Mall          25
10                    Residence Hall/Dormitory          21
11                                  Laboratory          17
12       Hospital (General Medical & Surgical)          16
13                                       Other          16
14                              Medical Office          14
15              Fitness Center/Health Club/Gym          

In [5]:
def building_dot_map(df):
    chart = alt.Chart(df).mark_circle(opacity=0.6).encode(
        longitude="longitude:Q",
        latitude="latitude:Q",
        size=alt.Size("total_ghg_emissions_metric_tons_co2e_", scale=alt.Scale(range=[5,60], type="sqrt"),
                        title="Total GHG emissions (metric tons)"),
        tooltip=["property_name", "primary_property_type",
                     alt.Tooltip("ghg_total:Q", format=",.0f"), "chicago_energy_rating"],
    ).project("mercator", center=[-87.68, 41.84], scale=62000).properties(width=500, height=600, title="Buildings by emissions and rating")

    return chart

building_dot_map(latest_top).save("draft_charts/1_building_dot_map.png")

In [6]:
def building_choropleth(df, metric):
    community_area = (df.assign(ca_key=df["community_area"].astype(str).str.upper().str.strip())
                        .groupby("ca_key")
                        .agg(count=("id", "count"),
                             median=("#_of_buildings", "median"),
                             sum=("#_of_buildings", "sum"))
                        .reset_index())
    labels = {"count": "Reporting buildings", "median": "Median # of buildings per property",
              "sum": "Total buildings"}
    chart = alt.Chart(alt.Data(values=CHI_GJ["features"])).mark_geoshape(stroke="white", strokeWidth=0.5).transform_lookup(
            lookup="properties.community",
            from_=alt.LookupData(community_area, "ca_key", ["count", "median", "sum"]),
        ).encode(
            color=alt.Color(f"{metric}:Q", title=labels[metric], scale=alt.Scale(scheme="blues")),
        ).project("mercator").properties(width=450, height=550, title=f"{labels[metric]} by community area ({LATEST_YEAR})")
    return chart

building_choropleth(latest_top, metric="median").save("draft_charts/2.1_building_choropleth.png")
building_choropleth(latest_top, metric="count").save("draft_charts/2.2_building_choropleth.png")
building_choropleth(latest_top, metric="sum").save("draft_charts/2.3_building_choropleth.png")

In [7]:
def ghg_choropleth(df):
    community_area = (df.assign(ca_key=latest["community_area"].astype(str).str.upper().str.strip())
                .groupby("ca_key")
                .agg(median_ghg_intensity=("ghg_intensity_kg_co2e_sq_ft_", "median"),
                     buildings=("id", "count"))
                .reset_index())
    chart = alt.Chart(alt.Data(values=CHI_GJ["features"])).mark_geoshape(stroke="white", strokeWidth=0.5).transform_lookup(
                lookup="properties.community",
                from_=alt.LookupData(community_area, "ca_key", ["median_ghg_intensity", "buildings"]),
            ).encode(
                color=alt.Color("median_ghg_intensity:Q", title="Median GHG intensity (kg CO2e / sq ft)",
                                scale=alt.Scale(scheme="orangered")),
            ).project("mercator").properties(width=450, height=550,
                        title=f"Where are the least efficient buildings? ({LATEST_YEAR})")
    return chart

ghg_choropleth(latest_top).save("draft_charts/3_ghg_choropleth.png")

In [8]:
def emissions_by_type_bar_chart(df):
    property_type = (df.groupby("primary_property_type")
                       .agg(total_ghg=("total_ghg_emissions_metric_tons_co2e_", "sum"),
                            buildings=("id", "count"))
                       .reset_index()
                       .nlargest(15, "total_ghg"))
    chart = alt.Chart(property_type).mark_bar().encode(
                x=alt.X("total_ghg:Q", title="Total GHG emissions (metric tons CO2e)"),
                y=alt.Y("primary_property_type:N", sort="-x", title=None),
                color=alt.value("#c0392b"),
            ).properties(width=500, height=380,
                         title=f"Which property types emit the most? ({LATEST_YEAR})")
    return chart

emissions_by_type_bar_chart(latest).save("draft_charts/4_emissions_by_type.png")

In [9]:
def emissions_by_type_box_plot(df):
    d = df.dropna(subset=["ghg_intensity_kg_co2e_sq_ft_"])
    chart = alt.Chart(d).mark_boxplot(extent=1.5).encode(
                x=alt.X("ghg_intensity_kg_co2e_sq_ft_:Q", title="GHG intensity (kg CO2e / sq ft)"),
                y=alt.Y("primary_property_type:N", title=None,
                        sort=alt.EncodingSortField("ghg_intensity_kg_co2e_sq_ft_", op="median", order="descending")),
                color=alt.Color("primary_property_type:N", legend=None),
            ).properties(width=500, height=400,
                         title=f"Variation in emissions within building types ({LATEST_YEAR})")
    return chart

emissions_by_type_box_plot(latest_top).save("draft_charts/5_emissions_type_box.png")

In [10]:
def building_age_energy_scatter(df):
    d = df.dropna(subset=["site_eui_kbtu_sq_ft_", "year_built"])
    d = d[(d["year_built"] > 1850) & (d["site_eui_kbtu_sq_ft_"] < d["site_eui_kbtu_sq_ft_"].quantile(0.99))]
    pts = alt.Chart(d).mark_circle(size=35, opacity=0.5).encode(
                x=alt.X("year_built:Q", title="Year built", scale=alt.Scale(zero=False)),
                y=alt.Y("site_eui_kbtu_sq_ft_:Q", title="Site EUI (kBtu / sq ft)"),
                color=alt.Color("chicago_energy_rating:O", title="Stars", scale=alt.Scale(scheme="redyellowgreen")),
            )
    trend = alt.Chart(d).transform_loess("year_built", "site_eui_kbtu_sq_ft_").mark_line(color="black", size=3).encode(
                x="year_built:Q", y="site_eui_kbtu_sq_ft_:Q")
    chart = (pts + trend).properties(width=550, height=380,
                                     title=f"Do older buildings use more energy per square foot? ({LATEST_YEAR})")
    return chart

building_age_energy_scatter(latest).save("draft_charts/6_building_age_energy_scatter.png")

In [11]:
def energy_rating_stacked_bar(df):
    d = df.dropna(subset=["chicago_energy_rating"])
    ratings = (d.groupby(["primary_property_type", "chicago_energy_rating"])
                .agg(buildings=("id", "count"))
                .reset_index())
    ratings["share"] = ratings["buildings"] / ratings.groupby("primary_property_type")["buildings"].transform("sum")
    type_order = (d.groupby("primary_property_type")["chicago_energy_rating"]
                   .mean().sort_values(ascending=False).index.tolist())
    chart = alt.Chart(ratings).mark_bar().encode(
                x=alt.X("share:Q", title="Share of buildings", axis=alt.Axis(format="%")),
                y=alt.Y("primary_property_type:N", title=None, sort=type_order),
                color=alt.Color("chicago_energy_rating:O", title="Stars", scale=alt.Scale(scheme="redyellowgreen")),
            ).properties(width=500, height=380,
                         title=f"Chicago Energy Rating mix by property type ({LATEST_YEAR})")
    return chart

energy_rating_stacked_bar(latest_top).save("draft_charts/7_energy_rating_stacked_bar.png")

In [16]:
def fuel_mix(df):
    fuel_use = (df.groupby("primary_property_type")[list(FUEL_COLS)].sum()
                  .rename(columns=FUEL_COLS)
                  .reset_index()
                  .melt("primary_property_type", var_name="fuel", value_name="kbtu"))
    fuel_use["share"] = fuel_use["kbtu"] / fuel_use.groupby("primary_property_type")["kbtu"].transform("sum")
    type_order = (fuel_use[fuel_use["fuel"] == "Natural gas"]
                  .sort_values("share", ascending=False)["primary_property_type"].tolist())
    fuel_order = ["Natural gas", "Electricity", "District steam", "District chilled water", "Other fuel"]

    chart = alt.Chart(fuel_use).mark_bar().encode(
                x=alt.X("share:Q", title="Share of energy use (kBtu)", axis=alt.Axis(format="%")),
                y=alt.Y("primary_property_type:N", title=None, sort=type_order),
                color=alt.Color("fuel:N", title="Fuel", sort=fuel_order, scale=alt.Scale(scheme="tableau10")),
                order=alt.Order("fuel_rank:Q"),
            ).transform_calculate(
                fuel_rank=f"indexof({fuel_order}, datum.fuel)"
            ).properties(width=500, height=380,
                         title=f"Fuel mix by property type ({LATEST_YEAR})")
    return chart

fuel_mix(latest_top).save("draft_charts/8_fuel_mix.png")

In [17]:
def same_building_trend(df, start=2018, end=2023):
    metrics = {"site_eui_kbtu_sq_ft_": "Site EUI (energy used)",
               "ghg_intensity_kg_co2e_sq_ft_": "GHG intensity (emissions)"}
    d = df[df["data_year"].between(start, end)]
    wide = d.pivot_table(index="id", columns="data_year", values=list(metrics))
    ids = wide.dropna().index                       # buildings with every metric in every year
    d = d[d["id"].isin(ids)]
    trend = d.groupby("data_year")[list(metrics)].median().rename(columns=metrics)
    trend = (trend / trend.loc[start] * 100).reset_index().melt(
        "data_year", var_name="measure", value_name="index")

    base = alt.Chart(trend).encode(
        x=alt.X("data_year:O", title="Year", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("index:Q", title=f"Median, indexed ({start} = 100)", scale=alt.Scale(zero=False)),
        color=alt.Color("measure:N", title=None))
    lines = base.mark_line(point=True, strokeWidth=3).encode(
        tooltip=[alt.Tooltip("data_year:O", title="Year"), alt.Tooltip("measure:N", title="Measure"),
                 alt.Tooltip("index:Q", title="Index", format=".1f")])
    labels = base.transform_filter(alt.datum.data_year == end).mark_text(
        align="left", dx=8, fontSize=12).encode(text=alt.Text("index:Q", format=".0f"))
    ref = alt.Chart(pd.DataFrame({"y": [100]})).mark_rule(strokeDash=[4, 4], color="gray").encode(y="y:Q")
    return (ref + lines + labels).properties(
        width=500, height=350,
        title=f"The same {len(ids):,} buildings from {start}–{end}")

same_building_trend(chi_bench_df()).save("draft_charts/9_same_building_trend.png")

In [15]:
def energy_area_type_heatmap(df, top_n_areas=20):
    top_ca = df["community_area"].value_counts().head(top_n_areas).index
    cell = (df[df["community_area"].isin(top_ca)]
              .groupby(["community_area", "primary_property_type"])
              .agg(median_site_eui=("site_eui_kbtu_sq_ft_", "median"),
                   buildings=("id", "count"))
              .reset_index())
    chart = alt.Chart(cell).mark_rect().encode(
                x=alt.X("primary_property_type:N", title=None, axis=alt.Axis(labelAngle=-45)),
                y=alt.Y("community_area:N", title=None),
                color=alt.Color("median_site_eui:Q", title="Median site EUI (kBtu / sq ft)",
                                scale=alt.Scale(scheme="viridis")),
            ).properties(width=500, height=450,
                         title=f"Energy use by neighborhood and building type ({LATEST_YEAR})")
    return chart

energy_area_type_heatmap(latest_top).save("draft_charts/10_heatmap.png")